## XTTS-v2 (Coqui) - Voice Cloning Batch

Replica el flujo de `QwenTTS_Voice_Cloning.ipynb` usando **XTTS-v2** (multilingue, espanol nativo).

- Mismas rutas de entrada (`Small_variants/...`) y mismas oraciones objetivo.
- Salida en `TTS_output_XTTS/...`.
- XTTS-v2 **no usa `ref_text`**: clona desde `speaker_wav` + texto objetivo + `language`. Por eso **no hace falta Whisper** en este notebook.
- Pesos bajo licencia CPML (uso no comercial / investigacion).

In [ ]:
# Verificar GPU / CUDA
import torch

if torch.cuda.is_available():
    print("CUDA disponible. Detalles de la GPU:")
    print(f"  Numero de GPUs: {torch.cuda.device_count()}")
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  CUDA: {torch.version.cuda}")
    props = torch.cuda.get_device_properties(0)
    print(f"  Memoria total: {props.total_memory / (1024**3):.2f} GB")
else:
    print("CUDA no disponible. Active el runtime con GPU (T4) en Colab.")

In [ ]:
# Instalar Coqui TTS (fork mantenido, compatible con Python 3.12 / torch 2.x)
!pip install -q coqui-tts

> Si `pip` cambia `torch`/`numpy`/`transformers`, reinicie el entorno (*Runtime -> Restart session*) y siga desde la celda de imports.

In [ ]:
import os
import torch

# Aceptar la licencia del modelo (CPML) de forma no interactiva
os.environ["COQUI_TOS_AGREED"] = "1"

# PyTorch >= 2.6 usa weights_only=True por defecto y bloquea las clases de config de XTTS.
# Las habilitamos para que cargue el checkpoint.
try:
    from TTS.tts.configs.xtts_config import XttsConfig
    from TTS.tts.models.xtts import XttsAudioConfig, XttsArgs
    from TTS.config.shared_configs import BaseDatasetConfig
    torch.serialization.add_safe_globals([XttsConfig, XttsAudioConfig, BaseDatasetConfig, XttsArgs])
except Exception as e:
    print("Aviso (safe_globals):", e)
    os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

from TTS.api import TTS

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Cargando XTTS-v2...")
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)

In [ ]:
sentences = [
    "¿Sabías que el cuerpo humano posee sensores? ¡Sí! ¡Como los robots!",
    "Esos sensores se encargan de capturar la energía que nos rodea.",
    "La transforman y la transportan como sensaciones que llegan al cerebro.",
    "Los sensores más conocidos detectan formas y colores.",
    "También detectan sonidos, olores y gustos.",
    "Además producen muchas sensaciones en la piel, como frío y calor.",
    "¿Hay otros sensores? ¡Claro que sí!",
    "Muchos más son menos conocidos, pero también importantes.",
    "Algunos sensores nos mantienen en equilibrio. Otros actúan cuando comemos algo picante.",
    "Cuando las sensaciones llegan al cerebro, se encuentran con conocimientos guardados en la memoria.",
    "Esos conocimientos se acumulan durante años .Entonces se produce la percepción.",
    "La percepción nos dice qué significa la sensación. También nos dice si hay un mensaje.",
    "Desde pequeños aprendemos los sonidos de nuestra lengua.",
    "Si oímos una palabra que nunca escuchamos, tratamos de acercarla a una conocida.",
    "Si eso falla, podemos decir: ¡no la entendí!"
]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Generacion - una variante por corrida

Igual que en el notebook de Qwen: procese una variante cambiando el ultimo segmento de la ruta. La salida solo cambia el nombre del TTS en la carpeta raiz (`TTS_output_XTTS`).

In [ ]:
folder_variants = "/content/drive/MyDrive/Tesis experiments/Small_variants/NoDenoising_DNSMOS_2-7"
output_folder   = "/content/drive/MyDrive/Tesis experiments/TTS_output_XTTS/NoDenoising_DNSMOS_2-7"
count_sentence = 0

os.makedirs(output_folder, exist_ok=True)
audio_files = os.listdir(folder_variants)
for audio in audio_files:
    infer_sentence = sentences[count_sentence % len(sentences)]
    count_sentence += 1
    ref_audio_path = os.path.join(folder_variants, audio)

    name_wav = audio.split(".")[0] + ".wav"
    output_path = os.path.join(output_folder, name_wav)

    # XTTS-v2 clona desde el audio de referencia, sin ref_text
    tts.tts_to_file(
        text=infer_sentence,
        speaker_wav=ref_audio_path,
        language="es",
        file_path=output_path,
    )

### (Opcional) Barrer todas las variantes
Procesa de una sola vez todas las subcarpetas de `Small_variants/`.

In [ ]:
# OPCIONAL: barrer TODAS las variantes de Small_variants
base_variants = "/content/drive/MyDrive/Tesis experiments/Small_variants"
base_output   = "/content/drive/MyDrive/Tesis experiments/TTS_output_XTTS"

for config in sorted(os.listdir(base_variants)):
    in_dir = os.path.join(base_variants, config)
    if not os.path.isdir(in_dir):
        continue
    out_dir = os.path.join(base_output, config)
    os.makedirs(out_dir, exist_ok=True)

    count_sentence = 0
    for audio in sorted(os.listdir(in_dir)):
        infer_sentence = sentences[count_sentence % len(sentences)]
        count_sentence += 1
        ref_audio_path = os.path.join(in_dir, audio)

        name_wav = audio.split(".")[0] + ".wav"
        output_path = os.path.join(out_dir, name_wav)
        tts.tts_to_file(
            text=infer_sentence,
            speaker_wav=ref_audio_path,
            language="es",
            file_path=output_path,
        )
    print(f"[ok] {config}: {count_sentence} audios")

Una vez generados los audios en la carpeta `TTS_output_*`, evaluelos con **NISQA** (estandar y fine-tune en espanol) y vuelque los MOS en `data_mos.py`, igual que con QwenTTS.